#Filtering digital devices from metadata of Electronics

In [ ]:
from google.cloud import bigquery
import pandas as pd

PROJECT_ID = "cs163-project-487801"
TABLE_ID   = "cs163-project-487801.amazon_electronics.ml_sample2"

client = bigquery.Client(project=PROJECT_ID)

query = f"""
SELECT
  title,
  label
FROM `{TABLE_ID}`
WHERE title IS NOT NULL and label is not NULL
"""
df = client.query(query).to_dataframe()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8935 entries, 0 to 8934
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   title   8935 non-null   object
 1   label   8935 non-null   Int64 
dtypes: Int64(1), object(1)
memory usage: 148.5+ KB


In [ ]:
#inspect the sample data
df["label"].value_counts(normalize=True)

,proportion
label,
0,0.516508
1,0.483492


#Train/test split

In [ ]:
from sklearn.model_selection import train_test_split

x = df["title"]
y = df["label"]

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42,
    stratify=y if y.nunique() > 1 else None
)

#Pipeline (TF-IDF + Logistic Regression)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        min_df= 5,
        token_pattern=r'\b[a-zA-Z][a-zA-Z]+\b'
    )),
    ("clf", LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    ))
])

model

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(min_df=5, ngram_range=(1, 2),
                                 stop_words='english',
                                 token_pattern='\\b[a-zA-Z][a-zA-Z]+\\b')),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=2000))])

#Train + Evaluate

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

model.fit(x_train, y_train)
pred = model.predict(x_test)

print("Confusion matrix:\n", confusion_matrix(y_test, pred))
print("\nClassification report:\n", classification_report(y_test, pred, digits=4))



Confusion matrix:
 [[888  35]
 [ 17 847]]

Classification report:
               precision    recall  f1-score   support

         0.0     0.9812    0.9621    0.9716       923
         1.0     0.9603    0.9803    0.9702       864

    accuracy                         0.9709      1787
   macro avg     0.9708    0.9712    0.9709      1787
weighted avg     0.9711    0.9709    0.9709      1787



In [ ]:
probs = model.predict_proba(x_test)[:,1]
pd.Series(probs).describe()

,0
count,1787.000000
mean,0.491299
std,0.420869
min,0.002004
25%,0.054225
50%,0.416654
75%,0.932303
max,0.997440


In [ ]:
errors = x_test[(y_test==1) & (pred==0)]
errors.tail(40)

,title
3722,Panasonic Toughbook CF-31 MK5
3375,DELL Optiplex with 20-Inch Monitor
12,"Gaming Desktop, Multimedia Keyboard, Optical M..."
3862,"Scythe Katana 5 Air CPU Cooler, 92mm Single Tower"
2952,Intel Core i5-4690K Processor 3.9 4 BX80646I54...
1340,Samsung Galaxy Tab 2 10.1 16GB Titanium Silver...
1392,Samsung NP780Z5E-S01UB 15-Inch Laptop(i7-3635Q...
1321,Google Pixel 9a with Gemini - Unlocked Android...
2435,Apple New Total Wireless Prepaid - Apple iPhon...
3617,Intel BX80617I5560M Core i5-560M 2.66GHz Mobil...


In [ ]:
# top feature
vectorizer = model.named_steps["tfidf"]
vectorizer.get_feature_names_out()[:50]

array(['aa', 'aa aaa', 'aaa', 'aba', 'aba renewed', 'abs', 'absorbing',
       'absorption', 'abys', 'ac', 'ac adapter', 'ac bluetooth',
       'ac charger', 'ac dc', 'ac power', 'access', 'access point',
       'accessories', 'accessory', 'accessory usa', 'acer', 'acer aspire',
       'acer chromebook', 'acer desktop', 'acer extensa', 'acer iconia',
       'acer inch', 'acer intel', 'acer laptop', 'acer nitro',
       'acer predator', 'acer spin', 'acer swift', 'acer switch',
       'acer tablet', 'acer travelmate', 'acrylic', 'action',
       'action camera', 'activated', 'active', 'active noise', 'adapter',
       'adapter cable', 'adapter card', 'adapter charger',
       'adapter compatible', 'adapter converter', 'adapter cord',
       'adapter dongle'], dtype=object)

Note: As inspect top 50 features, having features like (00', '00 ghz', '000', '0000', '00001', '0001', '000gb','000gb 1tb') - Those are number of hardware specs, not help in product signals

In [ ]:
#most important fearture for digital and non-digital

import numpy as np

feature_names = vectorizer.get_feature_names_out()
coefs = model.named_steps["clf"].coef_[0]

top_pos = np.argsort(coefs)[-40:]
top_neg = np.argsort(coefs)[:30]

print("Digital indicators:")
print(feature_names[top_pos])

print("Non-digital indicators:")
print(feature_names[top_neg])

Digital indicators:
['emmc' 'unlocked' 'wifi' 'core ssd' 'newest' 'pro' 'vaio' 'computer' 'pc'
 'business' 'fi' 'ultrabook' 'intel' 'apple macbook' 'wi fi' 'hd' 'wi'
 'win' 'windows pro' 'gb' 'hdd' 'touch' 'gaming laptop' 'renewed' 'tablet'
 'asus' 'gaming' 'fhd' 'dell' 'touchscreen' 'lenovo' 'core' 'desktop'
 'apple' 'acer' 'ssd' 'ram' 'windows' 'hp' 'laptop']
Non-digital indicators:
['case' 'adapter' 'compatible' 'battery' 'usb' 'cover' 'replacement'
 'keyboard' 'cable' 'stand' 'bag' 'power' 'monitor' 'pack' 'sleeve'
 'charger' 'motherboard' 'wireless' 'tv' 'protector' 'screen' 'video'
 'speaker' 'mouse' 'laptop battery' 'mount' 'kit' 'skin'
 'desktop processors' 'headset']


#Save the model

In [ ]:
import joblib

MODEL_PATH = "/mnt/digital_device_.joblib"
joblib.dump(model, MODEL_PATH)
MODEL_PATH

'/mnt/digital_device_.joblib'

#Apply model to the dataset

##Apply model on entire Metadata

In [ ]:
from google.cloud import bigquery
client = bigquery.Client(project="cs163-project-487801")

query = """
SELECT
 main_category, title, parent_asin, average_rating, rating_number, price
FROM `cs163-project-487801.amazon_electronics.meta_Electronics`
WHERE title IS NOT NULL
AND main_category IN ('Computers', 'All Electronics', 'Amazon Devices', 'Office Products')

"""

df = client.query(query).to_dataframe()
df.head()

,main_category,title,parent_asin,average_rating,rating_number,price
0,All Electronics,2Set Black 6AA Battery Holder Case Storage Box...,B07FSBSMNF,2.6,2,NaN
1,Computers,256GB USB Flash Drive Rugged Waterproof Flash ...,B07NP17GSW,1.8,2,NaN
2,Computers,Dell Latitude E5430 14-Inch Intel Core i3 250G...,B06XNJ71JX,1.4,2,NaN
3,All Electronics,iPearl High Grade Silicone Keyboard Skin Cover...,B007SI738Y,1.5,2,NaN
4,All Electronics,"WiFi Range Extender, WiFi Signal Booster up to...",B093CWQR21,2.1,9,NaN


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 824916 entries, 0 to 824915
Data columns (total 6 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   main_category   824916 non-null  object 
 1   title           824916 non-null  object 
 2   parent_asin     824916 non-null  object 
 3   average_rating  824916 non-null  float64
 4   rating_number   824916 non-null  Int64  
 5   price           275555 non-null  float64
dtypes: Int64(1), float64(2), object(3)
memory usage: 38.5+ MB


In [ ]:
#Applied model to the dataset
df["prob_digital"] = model.predict_proba(df["title"])[:,1]
df["pred_label"] = model.predict(df["title"])

#threshold to reduce noise, 80% confident
threshold = 0.799
df["pred_label"] = (df["prob_digital"] >= threshold).astype(int)

#Result table
results = df[[
    "main_category",
    "title",
    "parent_asin",
    "average_rating",
    "rating_number",
    "price",
    "prob_digital",
    "pred_label"
]]

In [ ]:
table_id = "cs163-project-487801.amazon_digital_devices_cleaned.metadata_digital_device_result"

job = client.load_table_from_dataframe(
    results,
    table_id,
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
)

job.result()

LoadJob<project=cs163-project-487801, location=US, id=484cd9a5-e3c8-4341-b80c-6b462d3638e4>